In [25]:
using LimberJack
import LimberJack.TkEisHu

In [26]:
using Interpolations

# Utility: drag-epoch sound horizon from the EH fitting formula (same as inside TkEisHu).
# Returns r_d in Mpc.
function r_drag_EH(cpar::CosmoPar)
    wm = cpar.Ωm * cpar.h^2
    wb = cpar.Ωb * cpar.h^2
    keq    = (7.46e-2) * wm / (cpar.h * cpar.θCMB^2)   # Mpc⁻¹
    zeq    = (2.5e4)   * wm * (cpar.θCMB^-4)
    b1     = 0.313 * (wm^-0.419) * (1 + 0.607 * wm^0.674)
    b2     = 0.238 * wm^0.223
    zd     = 1291 * ((wm^0.251) / (1 + 0.659 * wm^0.828)) * (1 + b1 * wb^b2)
    R_pref = 31.5 * wb * (cpar.θCMB^-4)
    Rd     = R_pref * ((zd + 1) / 1e3)^-1
    Req    = R_pref * (zeq    / 1e3)^-1
    rs = sqrt(1 + Rd) + sqrt(Rd + Req)
    rs /= (1 + sqrt(Req))
    rs  = log(rs)
    rs *= (2 / (3 * keq)) * sqrt(6 / Req)
    return rs  # Mpc
end

"""
    m_shapefit(cosmo, k, k_pivot, Pk_pri_fid, Pk_EH_fid)

ShapeFit slope parameter m (Eq. 5.16 of arXiv:2404.07269).

d/d(ln k) of ln[(P_EH/P_prim) / (P_EH_fid/P_prim_fid)] evaluated at k_pivot.

- `k`          : wavenumbers in h Mpc⁻¹  (log-uniformly spaced, e.g. cosmo.settings.ks)
- `k_pivot`    : pivot scale in h Mpc⁻¹  (≈ 0.03 h Mpc⁻¹)
- `Pk_pri_fid` : A_s^fid · (k/k_pivot)^(n_s^fid − 1)
- `Pk_EH_fid`  : T²_EH(k) at fiducial cosmology (from TkEisHu)
"""
function m_shapefit(
    cosmo      :: Cosmology,
    k          :: Vector{Float64},
    k_pivot    :: Float64,
    Pk_pri_fid :: Vector{Float64},
    Pk_EH_fid  :: Vector{Float64},
)
    cpar      = cosmo.cpar
    Pk_pri    = @. cpar.As * (k / k_pivot)^(cpar.ns - 1)
    Pk_EH     = LimberJack.TkEisHu(cpar, k ./ cpar.h)
    log_ratio = @. log(Pk_EH / Pk_pri) - log(Pk_EH_fid / Pk_pri_fid)
    # cubic_spline_interpolation requires an AbstractRange for the knots
    lnk = range(log(k[1]), log(k[end]), length = length(k))
    itp = cubic_spline_interpolation(lnk, log_ratio, extrapolation_bc = Line())
    δ = 1e-5
    return (itp(log(k_pivot) + δ) - itp(log(k_pivot) - δ)) / (2δ)
end

"""
    A_shapefit(cosmo, k, k_pivot, Pk_pri_fid, Pk_EH_fid, r_d, r_d_fid)

ShapeFit amplitude A = A_sp / A_sp^fid (Eq. 5.15 of arXiv:2404.07269;
Eq. 4.11 of arXiv:2411.12021).

A_sp = (r_d^fid / r_d)³ · P_nw(k_p · r_d^fid / r_d) / P_nw^fid(k_p)

- `r_d`     : drag-epoch sound horizon of the model cosmology (Mpc)
- `r_d_fid` : drag-epoch sound horizon of the fiducial cosmology (Mpc)
"""
function A_shapefit(
    cosmo      :: Cosmology,
    k          :: Vector{Float64},
    k_pivot    :: Float64,
    Pk_pri_fid :: Vector{Float64},
    Pk_EH_fid  :: Vector{Float64},
    r_d        :: Float64,
    r_d_fid    :: Float64,
)
    cpar     = cosmo.cpar
    Pk_EH    = LimberJack.TkEisHu(cpar, k ./ cpar.h)
    Pk_model = @. cpar.As * (k / k_pivot)^(cpar.ns - 1) * Pk_EH   # proxy for P_nw
    Pk_fid   = @. Pk_pri_fid * Pk_EH_fid

    lnk       = range(log(k[1]), log(k[end]), length = length(k))
    itp_model = cubic_spline_interpolation(lnk, log.(Pk_model), extrapolation_bc = Line())
    itp_fid   = cubic_spline_interpolation(lnk, log.(Pk_fid),   extrapolation_bc = Line())

    k_eval = k_pivot * r_d_fid / r_d
    return (r_d_fid / r_d)^3 * exp(itp_model(log(k_eval))) / exp(itp_fid(log(k_pivot)))
end

"""
    fs8_shapefit(cosmo, z, k, k_pivot, Pk_pri_fid, Pk_EH_fid, r_d, r_d_fid, sigma_s8_fid; a_sf)

ShapeFit f σ_{s8} (Eq. 5.14 of arXiv:2404.07269; Eq. 4.10 of arXiv:2411.12021).

σ_{s8} = σ_{s8}^fid · A^{1/2} · exp[ m/(2a) · tanh(a · ln(r_d^fid / s_8)) ]

with s_8 = 8 h⁻¹ Mpc (model h).

- `r_d`          : model drag-epoch sound horizon (Mpc)
- `r_d_fid`      : fiducial drag-epoch sound horizon (Mpc)
- `sigma_s8_fid` : σ_8 of the fiducial cosmology
- `a_sf`         : ShapeFit 'a' parameter (default 0.6)
"""
function fs8_shapefit(
    cosmo        :: Cosmology,
    z            :: Float64,
    k            :: Vector{Float64},
    k_pivot      :: Float64,
    Pk_pri_fid   :: Vector{Float64},
    Pk_EH_fid    :: Vector{Float64},
    r_d          :: Float64,
    r_d_fid      :: Float64,
    sigma_s8_fid :: Float64;
    a_sf         :: Float64 = 0.6,
)
    cpar = cosmo.cpar
    m    = m_shapefit(cosmo, k, k_pivot, Pk_pri_fid, Pk_EH_fid)
    A    = A_shapefit(cosmo, k, k_pivot, Pk_pri_fid, Pk_EH_fid, r_d, r_d_fid)

    s8_Mpc   = 8.0 / cpar.h
    sigma_s8 = sigma_s8_fid * sqrt(A) *
               exp(m / (2 * a_sf) * tanh(a_sf * log(r_d_fid / s8_Mpc)))

    fz = cosmo.fs8z(z) / (cpar.σ8 * cosmo.Dz(z))
    return fz * sigma_s8
end

fs8_shapefit

In [27]:
settings  = cosmo.settings
cpar_fid  = cosmo.cpar

k       = settings.ks    # h/Mpc
k_pivot = 0.03           # h/Mpc  (≈ π / r_d^fid for typical ΛCDM)

Pk_EH_fid  = LimberJack.TkEisHu(cpar_fid, k ./ cpar_fid.h)
Pk_pri_fid = @. cpar_fid.As * (k / k_pivot)^(cpar_fid.ns - 1)

r_d_fid = r_drag_EH(cpar_fid)
r_d     = r_drag_EH(cosmo.cpar)    # same here; differs when model ≠ fiducial
println("r_d^fid = $(round(r_d_fid; digits=2)) Mpc")

# Self-consistency: model = fiducial → m≈0, A≈1, fs8_s8 ≈ fs8
m_val   = m_shapefit(cosmo, k, k_pivot, Pk_pri_fid, Pk_EH_fid)
A_val   = A_shapefit(cosmo, k, k_pivot, Pk_pri_fid, Pk_EH_fid, r_d, r_d_fid)
fs8_val = fs8_shapefit(cosmo, 0.5, k, k_pivot, Pk_pri_fid, Pk_EH_fid,
                        r_d, r_d_fid, cpar_fid.σ8)

println("m  (≈ 0): ", round(m_val;   digits=6))
println("A  (≈ 1): ", round(A_val;   digits=6))
println("fσ_s8 at z=0.5: ", round(fs8_val;        digits=6))
println("fσ_8  at z=0.5: ", round(cosmo.fs8z(0.5); digits=6))

r_d^fid = 102.6 Mpc
m  (≈ 0): 0.0
A  (≈ 1): 1.0
fσ_s8 at z=0.5: 0.465594
fσ_8  at z=0.5: 0.376405
